In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import pandas as pd
import numpy as np
import re
import nltk
import contractions
import unidecode

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

STOP_WORDS = set(stopwords.words('english'))
STOP_WORDS -= {'no', 'not', "n't", 'nor', 'never', 'nothing', 'nobody',
               'neither', 'nowhere', 'cannot'}

lemmatizer = WordNetLemmatizer()
print("Imports ready.")

Imports ready.


In [3]:
df = pd.read_csv(
    "/Users/vinhlam/PycharmProjects/smartphones-review-analyzer/data/processed/master_reviews.csv",
    low_memory=False,
    dtype={'review_title': str, 'review_date': str, 'verified_purchase': str}
)
print(f"Loaded {len(df):,} reviews")
print(df[['brand', 'rating', 'review_body']].head(3))

Loaded 137,388 reviews
     brand  rating                                        review_body
0  Samsung       5  I feel so LUCKY to have found this used (phone...
1  Samsung       4  nice phone, nice up grade from my pantach revu...
2  Samsung       5                                       Very pleased


In [4]:
def clean_text(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""
    
    text = unidecode.unidecode(text)
    text = text.lower()
    text = contractions.fix(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    return ' '.join(tokens)

def clean_for_vader(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Test it
sample = "I LOVE this phone!! Don't buy it if you can't afford the accessories :)"
print("Original:", sample)
print("Cleaned: ", clean_text(sample))

Original: I LOVE this phone!! Don't buy it if you can't afford the accessories :)
Cleaned:  love phone not buy not afford accessory


In [5]:
print("Cleaning review text... (this takes 2-5 minutes)")

df['review_clean'] = df['review_body'].apply(clean_text)
df['review_vader'] = df['review_body'].apply(clean_for_vader)

print(f"\nDone!")
print(f"Empty cleaned reviews: {(df['review_clean'] == '').sum()}")
print(f"\nSample before/after:")
for i in range(3):
    print(f"  Before: {df['review_body'].iloc[i][:100]}")
    print(f"  After:  {df['review_clean'].iloc[i][:100]}")
    print()

Cleaning review text... (this takes 2-5 minutes)

Done!
Empty cleaned reviews: 427

Sample before/after:
  Before: I feel so LUCKY to have found this used (phone to us & not used hard at all), phone on line from som
  After:  feel lucky found used phone not used hard phone line someone upgraded sold one son liked old one fin

  Before: nice phone, nice up grade from my pantach revue. Very clean set up and easy set up. never had an and
  After:  nice phone nice grade pantach revue clean set easy set never android phone fantastic say least perfe

  Before: Very pleased
  After:  pleased



In [6]:
df['word_count'] = df['review_clean'].str.split().str.len()
df['char_count'] = df['review_body'].str.len()

print("Word count stats:")
print(df['word_count'].describe())

before = len(df)
df = df[df['word_count'] >= 3].copy()
print(f"\nRemoved {before - len(df):,} too-short reviews")
print(f"Remaining: {len(df):,}")

print("\nAverage review length by brand:")
print(df.groupby('brand')['word_count'].mean().round(1).sort_values(ascending=False))

Word count stats:
count    137388.000000
mean         25.268946
std          49.432631
min           0.000000
25%           5.000000
50%          12.000000
75%          26.000000
max        2920.000000
Name: word_count, dtype: float64

Removed 13,280 too-short reviews
Remaining: 124,108

Average review length by brand:
brand
Nokia       46.2
Google      36.3
Huawei      36.2
Motorola    33.2
OnePlus     31.4
Xiaomi      28.2
Samsung     25.3
LG          24.7
Apple       16.7
Name: word_count, dtype: float64


In [7]:
def assign_price_tier(price):
    if pd.isna(price) or price <= 0:
        return 'Unknown'
    elif price < 100:
        return 'Budget (<$100)'
    elif price < 250:
        return 'Mid-Range ($100-$250)'
    elif price < 500:
        return 'Upper-Mid ($250-$500)'
    else:
        return 'Premium ($500+)'

df['price_tier'] = df['price'].apply(assign_price_tier)

print("Price tier distribution:")
print(df['price_tier'].value_counts())

print("\nPrice tier by brand (% of brand reviews):")
tier_by_brand = df.groupby(['brand', 'price_tier']).size().unstack(fill_value=0)
tier_pct = tier_by_brand.div(tier_by_brand.sum(axis=1), axis=0).round(3) * 100
print(tier_pct)

Price tier distribution:
price_tier
Mid-Range ($100-$250)    52420
Upper-Mid ($250-$500)    27694
Budget (<$100)           24747
Premium ($500+)           9825
Unknown                   9422
Name: count, dtype: int64

Price tier by brand (% of brand reviews):
price_tier  Budget (<$100)  Mid-Range ($100-$250)  Premium ($500+)  Unknown  \
brand                                                                         
Apple                 18.1                   46.0             14.9      0.6   
Google                 0.0                   65.3              7.1      6.2   
Huawei                 8.8                   56.2              5.3      0.5   
LG                    37.7                   34.9              4.0      0.6   
Motorola              30.6                   51.6              0.1      9.4   
Nokia                 31.0                   45.3              0.4      4.3   
OnePlus                0.0                    0.2             15.7      6.3   
Samsung               14.5   